# Week 1 Lab 2: The Complexity of Parallelizing Tasks

**ISM 6562 — Big Data for Business Applications**

Dr. Tim Smith

# Introduction

In Lab 1, we parallelized a simple task: each worker performed an independent computation and returned a result. The tasks didn't depend on each other, and combining the results was trivial.

But many real-world problems aren't like that. Some computations have **data dependencies** — where each result depends on previous results. Think of a running bank balance: you can't compute today's balance without knowing yesterday's.

In this lab, we'll explore a deceptively simple problem — computing a **running total** (cumulative sum) — and discover why it resists parallelization.

## Environment Setup

In [1]:
#!pip install -q multiprocess

In [2]:
import multiprocess
from multiprocess import Pool
import random

number_of_cores = multiprocess.cpu_count()
number_of_cores_to_use = max(1, number_of_cores - 1)

print(f"CPU cores available: {number_of_cores}")
print(f"Cores we'll use: {number_of_cores_to_use}")

CPU cores available: 16
Cores we'll use: 15


# Step 1: The Problem — Running Total

A **running total** (or cumulative sum) produces a new list where each element is the sum of all elements up to that point.

For example:

```
Input:         [5,  3,  8,  2,  7]
Running total: [5,  8, 16, 18, 25]
```

This shows up everywhere in business:

- **Bank account balance** — each transaction depends on the previous balance
- **Year-to-date revenue** — each month's YTD total includes all prior months
- **Inventory levels** — current stock depends on all previous shipments and sales

Let's generate some data to work with.

In [3]:
# Simulate daily transactions (positive = deposit, negative = withdrawal)
random.seed(42)
num_transactions = 20_000_000
transactions = [random.randint(-100, 200) for _ in range(num_transactions)]

print(f"Generated {len(transactions):,} transactions")
print(f"First 10: {transactions[:10]}")

Generated 20,000,000 transactions
First 10: [-43, -88, 40, 25, 14, -29, -48, 179, -56, 116]


# Step 2: Sequential Running Total

The sequential version is straightforward — walk through the list, keeping a running sum.

In [4]:
%%time
def sequential_running_total(data):
    result = []
    running_sum = 0
    for value in data:
        running_sum += value
        result.append(running_sum)
    return result

correct_result = sequential_running_total(transactions)

print(f"First 10 transactions:    {transactions[:10]}")
print(f"First 10 running totals:  {correct_result[:10]}")
print(f"Final balance:            {correct_result[-1]:,}")

First 10 transactions:    [-43, -88, 40, 25, 14, -29, -48, 179, -56, 116]
First 10 running totals:  [-43, -131, -91, -66, -52, -81, -129, 50, -6, 110]
Final balance:            999,687,766
CPU times: user 418 ms, sys: 219 ms, total: 637 ms
Wall time: 636 ms


# Step 3: Naive Parallel Attempt

Let's try the obvious approach: split the transactions into chunks, have each worker compute a running total for its chunk, then stitch the results together.

In [5]:
def chunk_running_total(chunk):
    """Compute running total for a single chunk, starting from 0."""
    result = []
    running_sum = 0
    for value in chunk:
        running_sum += value
        result.append(running_sum)
    return result

def split_into_chunks(data, num_chunks):
    """Split a list into num_chunks roughly equal pieces."""
    chunk_size = len(data) // num_chunks
    chunks = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = start + chunk_size if i < num_chunks - 1 else len(data)
        chunks.append(data[start:end])
    return chunks

In [6]:
# Split and process in parallel
chunks = split_into_chunks(transactions, number_of_cores_to_use)

with Pool(number_of_cores_to_use) as pool:
    chunk_results = pool.map(chunk_running_total, chunks)

# Stitch the chunk results together
naive_result = []
for chunk_result in chunk_results:
    naive_result.extend(chunk_result)

# Compare with the correct result
print(f"Correct final balance:  {correct_result[-1]:,}")
print(f"Naive parallel balance: {naive_result[-1]:,}")
print(f"Match: {correct_result == naive_result}")

Correct final balance:  999,687,766
Naive parallel balance: 66,658,484
Match: False


The results **do not match**. Let's look at a small example to see exactly why.

In [7]:
# Small example: 8 transactions split across 2 workers
small = [5, 3, 8, 2, 7, 1, 4, 6]

print("Correct running total:")
print(f"  Input:  {small}")
print(f"  Result: {sequential_running_total(small)}")
print()

# What each worker computes independently
chunk_a = small[:4]  # [5, 3, 8, 2]
chunk_b = small[4:]  # [7, 1, 4, 6]

print("Worker A (first half):")
print(f"  Input:  {chunk_a}")
print(f"  Result: {chunk_running_total(chunk_a)}")
print()

print("Worker B (second half):")
print(f"  Input:  {chunk_b}")
print(f"  Result: {chunk_running_total(chunk_b)}")
print()

print("Stitched together (WRONG):")
print(f"  Result: {chunk_running_total(chunk_a) + chunk_running_total(chunk_b)}")

Correct running total:
  Input:  [5, 3, 8, 2, 7, 1, 4, 6]
  Result: [5, 8, 16, 18, 25, 26, 30, 36]

Worker A (first half):
  Input:  [5, 3, 8, 2]
  Result: [5, 8, 16, 18]

Worker B (second half):
  Input:  [7, 1, 4, 6]
  Result: [7, 8, 12, 18]

Stitched together (WRONG):
  Result: [5, 8, 16, 18, 7, 8, 12, 18]


The problem is clear: **Worker B starts from 0**, but it should start from where Worker A left off. Worker B doesn't know Worker A's final total because the workers run independently — that's the whole point of parallelism.

This is a **data dependency**: every element in the running total depends on *all previous elements*. You can't compute position 5 without knowing the sum of positions 1–4, which means you can't start Worker B until Worker A is done... which defeats the purpose of parallelism.

# Step 4: Fixing It — A Two-Phase Approach

We can make it work, but it requires extra coordination:

**Phase 1:** Each worker computes its chunk's running total (starting from 0) *and* returns the chunk's final sum.

**Phase 2:** We compute the offset for each chunk (the total of all previous chunks), then add that offset to every element in the chunk.

This second phase can't be parallelized easily — it requires knowing the results of Phase 1 first.

In [8]:
def chunk_running_total_with_sum(chunk):
    """Return (running_total_list, chunk_sum) for a chunk."""
    result = []
    running_sum = 0
    for value in chunk:
        running_sum += value
        result.append(running_sum)
    return (result, running_sum)

In [9]:
%%time
# Phase 1: each worker computes its chunk's running total
chunks = split_into_chunks(transactions, number_of_cores_to_use)

with Pool(number_of_cores_to_use) as pool:
    phase1_results = pool.map(chunk_running_total_with_sum, chunks)

# Phase 2: compute offsets and adjust each chunk
# This part must be done sequentially — each offset depends on all previous chunks
corrected_result = []
offset = 0
for chunk_totals, chunk_sum in phase1_results:
    for value in chunk_totals:
        corrected_result.append(value + offset)
    offset += chunk_sum

print(f"Correct final balance:       {correct_result[-1]:,}")
print(f"Two-phase parallel balance:  {corrected_result[-1]:,}")
print(f"Match: {correct_result == corrected_result}")

Correct final balance:       999,687,766
Two-phase parallel balance:  999,687,766
Match: True
CPU times: user 11.5 s, sys: 1.33 s, total: 12.9 s
Wall time: 13.6 s


It works — but look at what we had to do:

1. Run Phase 1 in parallel (compute chunk running totals)
2. Wait for **all** workers to finish
3. Run Phase 2 **sequentially** (compute offsets and adjust every element)

Phase 2 loops through the entire dataset a second time. We're doing roughly **twice the work** of the sequential version, and Phase 2 can't be parallelized because each chunk's offset depends on the previous chunk's total.

# Step 5: Performance Comparison

Let's see whether the two-phase parallel approach is actually faster.

In [10]:
%%time
# Sequential
seq_result = sequential_running_total(transactions)
print(f"Sequential — final balance: {seq_result[-1]:,}")

Sequential — final balance: 999,687,766
CPU times: user 423 ms, sys: 212 ms, total: 635 ms
Wall time: 634 ms


In [11]:
%%time
# Two-phase parallel
chunks = split_into_chunks(transactions, number_of_cores_to_use)

with Pool(number_of_cores_to_use) as pool:
    phase1_results = pool.map(chunk_running_total_with_sum, chunks)

par_result = []
offset = 0
for chunk_totals, chunk_sum in phase1_results:
    for value in chunk_totals:
        par_result.append(value + offset)
    offset += chunk_sum

print(f"Parallel  — final balance: {par_result[-1]:,}")

Parallel  — final balance: 999,687,766
CPU times: user 11.5 s, sys: 2.01 s, total: 13.5 s
Wall time: 14.3 s


> **Understanding the `%%time` output:**
>
> - **CPU times: user** — time spent executing your Python code on the CPU
> - **CPU times: sys** — time spent on system-level operations (memory allocation, I/O, etc.)
> - **CPU times: total** — user + sys combined
> - **Wall time** — the actual elapsed real-world time from start to finish (what you'd measure with a stopwatch)

The parallel version is likely **slower** than the sequential version. The reasons:

- **Data copying overhead** — 20 million integers must be serialized and sent to each worker process, then the results sent back.
- **Phase 2 is sequential** — we still have to loop through every element to apply the offsets, and this can't be parallelized.
- **Double the work** — we effectively process each element twice (once in Phase 1, once in Phase 2).

The sequential version simply walks through the data once. For this type of problem, the straightforward approach wins.

# Summary

This lab demonstrated a fundamental challenge with parallelism: **not all algorithms can be easily parallelized**.

1. **Data dependencies block parallelism.** In a running total, each value depends on all previous values. Workers can't operate independently because they need information from each other.
2. **Fixing it adds complexity.** The two-phase approach works, but requires extra coordination — wait for all workers, compute offsets sequentially, then adjust every element. The code is more complex and does more total work.
3. **More workers doesn't always mean faster.** The overhead of splitting data, copying it between processes, and coordinating results can outweigh the benefit — especially when the per-element computation is cheap.

In Lab 1, we saw a task that was **embarrassingly parallel** — no dependencies between tasks, easy to split and combine. The running total is the opposite: a task where the dependencies are so tight that parallelism barely helps.

Most real-world Big Data problems fall somewhere in between. Recognizing where a problem sits on this spectrum — and choosing the right strategy — is a key skill in distributed computing.